In [1]:
import hashlib
import json
import math
import subprocess
import sys
import tempfile
from pathlib import Path

import torch
from torch import nn

# ==================================================
# PARAMÈTRES À ADAPTER
# ==================================================

# Si tu as attaché la version 1 du dataset checkpoints :
# EXPECTED_EXPERIMENT_ID = "EXP-001"

# Si tu as attaché la version 2 du dataset checkpoints :
# EXPECTED_EXPERIMENT_ID = "EXP-002"

EXPECTED_EXPERIMENT_ID = "EXP-002"

REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
GIT_REF = {
    "EXP-001": "exp/cae-bottle-exp001",
    "EXP-002": "exp/cae-bottle-exp002",
}[EXPECTED_EXPERIMENT_ID]

# On force CPU pour une vérification simple et stable.
device = torch.device("cpu")

print("Expected experiment:", EXPECTED_EXPERIMENT_ID)
print("Device:", device)

Expected experiment: EXP-001
Device: cpu


In [2]:
repo = Path(
    tempfile.mkdtemp(prefix="industrial-defect-ai-", dir="/tmp")
) / "repository"

subprocess.run(
    [
        "git", "clone", "--depth", "1",
        "--branch", GIT_REF,
        REPO_URL, str(repo),
    ],
    check=True,
)

sys.path.insert(0, str(repo))

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8-sig"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

repo_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo,
    text=True,
).strip()

print("Repository:", repo)
print("Repo commit:", repo_commit)

Cloning into '/tmp/industrial-defect-ai-nk3oe79c/repository'...


Repository: /tmp/industrial-defect-ai-nk3oe79c/repository
Repo commit: 52d4bbd839191e74d296892411062bc701da4aba


In [3]:
input_root = Path("/kaggle/input")

# Chercher le JSON de l'expérience attendue
metadata_matches = list(
    input_root.rglob(f"{EXPECTED_EXPERIMENT_ID}.json")
)

if len(metadata_matches) != 1:
    raise RuntimeError(
        f"{EXPECTED_EXPERIMENT_ID}: attendu 1 fichier JSON, "
        f"trouvé {len(metadata_matches)}."
    )

metadata_path = metadata_matches[0]
checkpoint_path = metadata_path.parent / "bottle_best.pth"

if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Checkpoint introuvable : {checkpoint_path}"
    )

# Dataset MVTec AD
dataset_root = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")

if not (dataset_root / "bottle/train/good").is_dir():
    raise FileNotFoundError(
        f"Dataset MVTec AD introuvable à {dataset_root}"
    )

metadata = read_json(metadata_path)

assert metadata["experiment_id"] == EXPECTED_EXPERIMENT_ID

print("Metadata:", metadata_path)
print("Checkpoint:", checkpoint_path)
print("Dataset root:", dataset_root)
print("Experiment loaded:", metadata["experiment_id"])

Metadata: /kaggle/input/cae-bottle-exp001-checkpoint/cae_bottle_EXP-001_bvufs8wr/EXP-001.json
Checkpoint: /kaggle/input/cae-bottle-exp001-checkpoint/cae_bottle_EXP-001_bvufs8wr/bottle_best.pth
Dataset root: /kaggle/input/datasets/ipythonx/mvtec-ad
Experiment loaded: EXP-001


In [4]:
# Lecture des champs importants
best_epoch = metadata["metrics"]["best_epoch"]
best_val_mse = metadata["metrics"]["best_val_mse"]
recorded_restored_val_mse = metadata["metrics"].get("restored_val_mse")
recorded_model_sha256 = metadata["artifacts"].get("model_sha256")

git_commit_from_metadata = metadata["reproducibility"]["git_commit"]

assert repo_commit == git_commit_from_metadata, (
    "Le commit restauré ne correspond pas au commit d'entraînement."
)

dataset_version = metadata["reproducibility"]["dataset_version"]

print("Best epoch:", best_epoch)
print("Best val MSE:", best_val_mse)
print("Recorded restored val MSE:", recorded_restored_val_mse)
print("Training git commit:", git_commit_from_metadata)
print("Dataset version:", dataset_version)

# Vérifier SHA-256 du checkpoint
actual_model_sha256 = sha256(checkpoint_path)

print("Recorded model SHA256:", recorded_model_sha256)
print("Actual model SHA256  :", actual_model_sha256)

if recorded_model_sha256 is not None:
    assert actual_model_sha256 == recorded_model_sha256, (
        "Le SHA-256 du checkpoint ne correspond pas au JSON."
    )

print("Intégrité du checkpoint : OK")

Best epoch: 25
Best val MSE: 0.002301636484584638
Recorded restored val MSE: 0.002301636484584638
Training git commit: 8fba62c63ad1bbb57b11eee84e3dfd27b23c7c77
Dataset version: 2
Recorded model SHA256: 85e5ca8406302ee7ccdd0b8ef83f273c7b41ab085c8b8d4bb5b6d0c36c8b5dde
Actual model SHA256  : 85e5ca8406302ee7ccdd0b8ef83f273c7b41ab085c8b8d4bb5b6d0c36c8b5dde
Intégrité du checkpoint : OK


In [5]:
from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform
from src.data.dataloaders import create_dataloader

from src.models.convolutional_autoencoder import (
    CAEConfig,
    ConvolutionalAutoencoder,
)

# Retrouver le config modèle
model_config_relpath = metadata["reproducibility"]["model_config"]
model_config_path = repo / model_config_relpath

model_config_json = read_json(model_config_path)["config"]
model_config_json["image_size"] = tuple(model_config_json["image_size"])
model_config_json["encoder_channels"] = tuple(
    model_config_json["encoder_channels"]
)

model_config = CAEConfig(**model_config_json)

# Retrouver le manifeste
manifest_relpath = metadata["reproducibility"]["split_manifest"]
manifest_path = repo / manifest_relpath

transform = MVTecTransform(
    size=model_config.image_size,
    profile="cae",
)

val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest_path,
    category="bottle",
    split="val",
    transform=transform,
)

assert len(val_dataset) == 42
assert all(sample["label"] == 0 for sample in val_dataset.samples)

val_loader = create_dataloader(
    val_dataset,
    batch_size=16,
    num_workers=0,
    seed=42,
)

print("Validation images:", len(val_dataset))
print("Validation batches:", len(val_loader))
print("Validation dataset : OK")

Validation images: 42
Validation batches: 3
Validation dataset : OK


In [6]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True,
)

restored_model = ConvolutionalAutoencoder(
    model_config
).to(device)

restored_model.load_state_dict(checkpoint["model_state_dict"])
restored_model.eval()

criterion = nn.MSELoss()

val_loss_sum = 0.0

with torch.inference_mode():
    for batch in val_loader:
        images = batch["image"].to(device)

        reconstructed = restored_model(images)
        loss = criterion(reconstructed, images)

        val_loss_sum += loss.item() * images.size(0)

recomputed_val_mse = val_loss_sum / len(val_dataset)

print("Best val MSE (metadata)      :", best_val_mse)
print("Restored val MSE (metadata)  :", recorded_restored_val_mse)
print("Recomputed val MSE (notebook):", recomputed_val_mse)

Best val MSE (metadata)      : 0.002301636484584638
Restored val MSE (metadata)  : 0.002301636484584638
Recomputed val MSE (notebook): 0.0023016363404513825


In [7]:
# Tolérance numérique raisonnable
# (petites différences possibles selon environnement)
reference_mse = (
    recorded_restored_val_mse
    if recorded_restored_val_mse is not None
    else best_val_mse
)

assert math.isclose(
    recomputed_val_mse,
    reference_mse,
    rel_tol=1e-4,
    abs_tol=1e-6,
), (
    "La MSE recalculée ne correspond pas à la valeur enregistrée."
)

print("Restauration indépendante : OK")
print(f"{EXPECTED_EXPERIMENT_ID} est reproductible sans réentraînement.")

Restauration indépendante : OK
EXP-001 est reproductible sans réentraînement.


In [8]:
output_dir = Path("/kaggle/working") / f"restore_check_{EXPECTED_EXPERIMENT_ID}"
output_dir.mkdir(parents=True, exist_ok=True)

report = {
    "experiment_id": EXPECTED_EXPERIMENT_ID,
    "checkpoint_path": str(checkpoint_path),
    "metadata_path": str(metadata_path),
    "repo_git_ref_used_for_restore": GIT_REF,
    "repo_commit_used_for_restore": repo_commit,
    "training_git_commit": git_commit_from_metadata,
    "dataset_version": dataset_version,
    "best_epoch": best_epoch,
    "best_val_mse": best_val_mse,
    "recorded_restored_val_mse": recorded_restored_val_mse,
    "recomputed_val_mse": recomputed_val_mse,
    "model_sha256": actual_model_sha256,
    "status": "OK",
}

report_path = output_dir / f"{EXPECTED_EXPERIMENT_ID}_restore_report.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Rapport :", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))

Rapport : /kaggle/working/restore_check_EXP-001/EXP-001_restore_report.json
{
  "experiment_id": "EXP-001",
  "checkpoint_path": "/kaggle/input/cae-bottle-exp001-checkpoint/cae_bottle_EXP-001_bvufs8wr/bottle_best.pth",
  "metadata_path": "/kaggle/input/cae-bottle-exp001-checkpoint/cae_bottle_EXP-001_bvufs8wr/EXP-001.json",
  "repo_branch_used_for_restore": "feature/us-03-02-cae-bottle",
  "repo_commit_used_for_restore": "52d4bbd839191e74d296892411062bc701da4aba",
  "training_git_commit": "8fba62c63ad1bbb57b11eee84e3dfd27b23c7c77",
  "dataset_version": "2",
  "best_epoch": 25,
  "best_val_mse": 0.002301636484584638,
  "recorded_restored_val_mse": 0.002301636484584638,
  "recomputed_val_mse": 0.0023016363404513825,
  "model_sha256": "85e5ca8406302ee7ccdd0b8ef83f273c7b41ab085c8b8d4bb5b6d0c36c8b5dde",
  "status": "OK"
}
